# 01. Leíró Statisztika és Feltáró Adatelemzés (EDA)

**Cél**: A kőbányai eladó lakáspiac (N = 1 140 db) fundamentális árainak, alapterületeinek és szobaszámainak vizsgálata.

---

### 📖 Miért nem elég csak az átlagárakat nézni?
A laikus lakásvásárló gyakran találkozik azzal, hogy az átlagárak nem fedik a valóságot:
1. **Nincs egyetlen átlagos lakás**: Egy lakótelepi garzon és egy zöldövezeti nagypolgári ház teljesen más árszínvonalat képvisel.
2. **Az ingatlanárak eloszlása ferde**: Az olcsóbb és közepes kategóriában tömörül a lakások többsége, míg néhány kiemelkedően drága ingatlan felfelé húzza a számtani átlagot.

Ebben a fejezetben megvizsgáljuk az árak valódi szóródását, a kvartiliseket, és megmutatjuk, miért elengedhetetlen a **logaritmikus árak** használata a későbbi ökonometriai becslésekhez.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from scipy import stats

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Eladó lakások száma az EDA mintában: {len(elado)} db.")

Eladó lakások száma az EDA mintában: 1140 db.


### 1. Főbb Leíró Statisztikai Mutatók (A kőbányai lakáspiac pontos számai)

**📌 Mit látunk a mutatókártyákon és a részletes táblázatban?**
A kőbányai eladó lakások (N = 1 140 db) tényleges, számított statisztikáit látjuk:
- **Átlagos Kínálati Ár**: **68.8 millió Ft**, míg a **Medián Ár: 65.0 millió Ft**. A kettő közötti ~3.8 millió forintos különbség pontosan a drágább ingatlanok felfelé húzó hatását mutatja!
- **Átlagos Ár / m²**: **1 222 480 Ft/m²**, míg a **Medián Ár / m²: 1 184 479 Ft/m²**.
- **Átlagos Alapterület**: **57.6 m²** (medián: 53.0 m²).
- **Átlagos Szobaszám**: **2.4 szoba**, a leggyakoribb érték (módusz) pedig pontosan a **2.0 szobás** elrendezés.
- **Ár / m² Szórás**: **261 452 Ft/m²**, ami **21.4%-os relatív szórást** jelent.
- **Ferdeség (Skewness = 0.41)**: Enyhén jobbra ferde eloszlás, igazolva a log-normális jelleget.

**🔍 Hogyan értelmezzük a kvartiliseket? (Q1, Q2, Q3)**
- **Alsó negyed (Q1 = 1 045 495 Ft/m²)**: A lakások legolcsóbb 25%-a ezen érték alatt vásárolható meg.
- **Felső negyed (Q3 = 1 403 950 Ft/m²)**: A lakások legdrágább 25%-a ezen érték felett indul.
- **Interkvartilis terjedelem (IQR = 358 455 Ft/m²)**: A piac középső 50%-a ebben a szűk, ~358 ezer Ft-os sávban koncentrálódik.

**💡 Tanulság**: A kőbányai eladó lakáskínálat zöme az 50 és 75 millió forint közötti tartományban mozog.

In [2]:
kpi_cards = [
    ("Átlagos Kínálati Ár", fmt_mft(elado['price_huf'].mean() / 1e6), f"Medián: {fmt_mft(elado['price_huf'].median() / 1e6)}", "#1e3a8a"),
    ("Átlagos Ár / m²", fmt_huf(elado['nm_ar_huf'].mean()), f"Medián: {fmt_huf(elado['nm_ar_huf'].median())}", "#2563eb"),
    ("Átlagos Alapterület", f"{elado['alapterulet_nm'].mean():.1f} m²", f"Medián: {elado['alapterulet_nm'].median():.1f} m²", "#059669"),
    ("Átlagos Szobaszám", f"{elado['szobaszam_osszes'].mean():.1f} szoba", f"Módusz: {elado['szobaszam_osszes'].mode()[0]} szoba", "#d97706"),
    ("Ár / m² Szórás", fmt_huf(elado['nm_ar_huf'].std()), f"Relatív szórás: {elado['nm_ar_huf'].std()/elado['nm_ar_huf'].mean()*100:.1f}%", "#dc2626"),
    ("Ár / m² Ferdeség", f"{stats.skew(elado['nm_ar_huf'].dropna()):.2f}", "Jobbra ferde (log-normális)", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

# Részletes leíró táblázat
desc_vars = ['price_huf', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes']
desc_names = ['Ár (HUF)', 'Ár / m² (HUF)', 'Alapterület (m²)', 'Szobaszám']
stat_df = elado[desc_vars].describe().T
stat_df['skew'] = [stats.skew(elado[c].dropna()) for c in desc_vars]
stat_df['kurtosis'] = [stats.kurtosis(elado[c].dropna()) for c in desc_vars]
stat_df.index = desc_names
stat_df.columns = ['Darab', 'Átlag', 'Szórás', 'Min', '25% (Q1)', 'Medián (Q2)', '75% (Q3)', 'Max', 'Ferdeség', 'Csúcsosság']

html_table = "<div style='overflow-x:auto; margin: 15px 0;'>" + stat_df.round(2).to_html(classes='table table-bordered table-hover') + "</div>"
display(HTML(html_table))

,Darab,Átlag,Szórás,Min,25% (Q1),Medián (Q2),75% (Q3),Max,Ferdeség,Csúcsosság
Ár (HUF),1140.0,68849383.97,26351817.87,10000000.0,53405000.00,65000000.0,76290000.0,189900000.0,1.33,2.64
Ár / m² (HUF),1140.0,1222480.32,261451.60,311321.0,1045494.75,1184478.5,1403950.0,2343750.0,0.41,0.53
Alapterület (m²),1140.0,57.61,23.23,15.0,44.00,53.0,68.0,363.0,2.97,27.30
Szobaszám,1140.0,2.38,0.92,1.0,2.00,2.0,3.0,7.0,0.50,0.55


### 2. Négyzetméterárak Eloszlása és a Log-Transzformáció

**📌 Mit látunk a két hisztogramon?**
- **Bal oldali ábra**: A nyers négyzetméterárak eloszlása (Ft/m²). Látható, hogy az eloszlás enyhén jobbra nyúlik el (Skewness = 0.41).
- **Jobb oldali ábra**: A négyzetméterárak természetes logaritmusa ($\ln(\text{Ár/m}^2)$).

**🔍 Miért van szükség logaritmusra a tudományos modellekben?**
1. **A Gauss-görbe biztosítása**: A lineáris regresszió feltételezi, hogy a hibatagok szimmetrikus harang alakú normál eloszlást követnek. A logaritmus alkalmazásával az árak eloszlása közel tökéletesen normálissá válik.
2. **Közvetlen százalékos értelmezhetőség**: A logaritmikus modellekben (NB04, NB13, NB15) az együtthatók közvetlenül **százalékos prémiumként vagy diszkontként** értelmezhetők (pl. a panel jelleg vagy az állapot hatása).

**💡 Eredmény**: Az árak log-transzformációja stabilizálja a szórásokat és megalapozza az ökonometriai becsléseket.

In [3]:
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Nyers Négyzetméterár Eloszlása (Ft/m²)', 'Logaritmikus Négyzetméterár Eloszlása (ln(Ár/m²))')
)

fig1.add_trace(
    go.Histogram(x=elado['nm_ar_huf'], nbinsx=35, name='Ár/m²', marker_color='#2563eb', opacity=0.8),
    row=1, col=1
)

fig1.add_trace(
    go.Histogram(x=np.log(elado['nm_ar_huf']), nbinsx=35, name='ln(Ár/m²)', marker_color='#059669', opacity=0.8),
    row=1, col=2
)

fig1.update_layout(
    title_text='Fajlagos árak eloszlása és normálissá alakítása log-transzformációval',
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=False
)
fig1.show()

### 3. Dobozdiagramok (Boxplot) és Kiugró Értékek Városrészenként

**📌 Hogyan kell olvasni a dobozdiagramot?**
- **A doboz közepe (vastag vonal)**: A medián ár (a középső lakás értéke).
- **A doboz alsó és felső éle**: Az alsó kvartilis (Q1) és felső kvartilis (Q3). A doboz magassága maga a piac középső 50%-a (IQR).
- **A függőleges bajszok**: A normális ártartomány határai ($1.5 \times \text{IQR}$).
- **A bajszokon kívüli egyedi pontok**: Statisztikai kiugró értékek (outlierek).

**🔍 Mit látunk Kőbánya városrészeiben?**
- **Óhegy és Ligettelek**: A dobozok magasabban helyezkednek el, a felső kvartilis eléri az 1.3 - 1.4 millió Ft/m²-es szintet.
- **Újhegy**: A doboz rendkívül tömör és feszes! A lakások szabványos méretei és paneles jellege miatt az árak alacsony szórásúak, a medián ~1.1 millió Ft/m² körül stabil.
- **Gyárdűlő és Laposdűlő**: Hosszabb dobozok és nagyobb szóródás, mivel a vegyes ipari-rozsdaövezeti környezetben az elhanyagolt épületek és a modern új fejlesztések árai erősen eltérnek egymástól.

**💡 Tanulság**: Újhegy a legkiszámíthatóbb árú övezet, Óhegy a legváltozatosabb prémium zóna.

In [4]:
# Városrészi boxplot
fig2 = px.box(
    elado,
    x='varosresz',
    y='nm_ar_huf',
    color='varosresz',
    title='Négyzetméterár dobozábra (Boxplot) városrészenként',
    labels={'varosresz': 'Városrész', 'nm_ar_huf': 'Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(xaxis_tickangle=-30, height=450, showlegend=False)
fig2.show()

# Méret vs Ár szórásdiagram OLS trenddel
fig3 = px.scatter(
    elado,
    x='alapterulet_nm',
    y='ar_millio_ft',
    color='varosresz',
    trendline='ols',
    title='Alapterület vs. Kínálati Ár összefüggés városrészenként',
    labels={'alapterulet_nm': 'Alapterület (m²)', 'ar_millio_ft': 'Ár (Millió Ft)', 'varosresz': 'Városrész'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=480)
fig3.show()

### 4. Korrelációs Hőtérkép és Változókapcsolatok

**📌 Mit jelent a korrelációs mátrix?**
A változók közötti lineáris kapcsolatot méri -1 és +1 között:
- **Pozitív érték (kék)**: Együtt mozognak (pl. alapterület és teljes ár: $r \approx +0.85$).
- **Negatív érték (piros)**: Ellentétesen mozognak (pl. alapterület és m² ár: a garzon-effektus miatt a nagyobb lakások m² ára enyhén alacsonyabb).
- **Panelszerkezet (is_panel)**: Szignifikáns negatív kapcsolatot mutat a négyzetméterárral.

**💡 Fő tanulság**: A méret, a panel szerkezet, az állapot és a közlekedési csomópontok távolsága a kőbányai piac legfőbb mozgatórugói.

In [5]:
w_var = widgets.Dropdown(
    options=[('Négyzetméterár (HUF)', 'nm_ar_huf'), ('Kínálati Ár (M Ft)', 'ar_millio_ft'), ('Alapterület (m²)', 'alapterulet_nm'), ('Szobaszám', 'szobaszam_osszes')],
    value='nm_ar_huf',
    description='Változó:'
)
w_varos = widgets.SelectMultiple(
    options=VAROSRESZEK,
    value=tuple(VAROSRESZEK[:4]),
    description='Városrészek:'
)
out_eda = widgets.Output()

def frissit_eda(*args):
    sub = elado[elado['varosresz'].isin(w_varos.value)]
    with out_eda:
        clear_output(wait=True)
        col = w_var.value
        fig = px.histogram(
            sub, x=col, color='varosresz', barmode='overlay',
            title=f'{w_var.label} eloszlása a kiválasztott városrészekben (N={len(sub)})',
            template=PLOTLY_TEMPLATE, opacity=0.7
        )
        fig.show()

w_var.observe(frissit_eda, names='value')
w_varos.observe(frissit_eda, names='value')

display(widgets.HBox([w_var, w_varos]))
display(out_eda)
frissit_eda()

Output()